# SEED-VII EEG Pipeline (单入口 Notebook)

**支持模型**：EEGNet (`--model-type eegnet`) | EEGConformer (`--model-type conformer`)

**数据源**：ModelScope `DEREKVERSE/SEED-VII` → `SEED-VII.npz`（预处理后）

**流程**：下载 → 训练 → 编码 → 结果汇总

---

⚠️ **只修改 Cell 2（配置区）即可切换实验**

In [ ]:
# 安装依赖（如已安装可跳过）
!pip install mne torch numpy scipy pandas tqdm modelscope -q

---

## ⚙️ Cell 2：实验配置（修改这里即可切换实验）

In [ ]:
# ================================================================
# 模型选择（必选）
# ================================================================
MODEL_TYPE = "eegnet"          # "eegnet" 或 "conformer"

# ================================================================
# ModelScope 数据配置
# ================================================================
MS_DATASET_ID   = "DEREKVERSE/SEED-VII"
MS_FILE_PATH    = "SEED-VII.npz"          # ModelScope 中预处理后 npz 的路径
MS_REVISION     = "master"
MS_TOKEN        = ""                        # 留空则用环境变量 MODELSCOPE_API_TOKEN
LOCAL_DATA_DIR  = "./data"                 # 本地缓存目录
LOCAL_DATA_PATH = f"{LOCAL_DATA_DIR}/SEED-VII.npz"  # 下载后本地文件

# ================================================================
# 训练超参数
# ================================================================
OUTPUT_DIR      = f"runs_seed_vii/{MODEL_TYPE}"
SEED            = 42
BATCH_SIZE      = 256
NUM_WORKERS     = 2
LR              = 2e-4
MIN_LR          = 1e-5
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
PRETRAIN_EPOCHS = 10        # 只训练分类头（关闭回归）的 epoch 数
MAX_EPOCHS      = 200
PATIENCE        = 30

# 损失权重
ALPHA_CLS       = 1.0       # 分类 loss 系数
BETA_REG        = 0.5       # 回归 loss 系数
ENABLE_RANK     = False     # 是否开启 margin ranking loss
RANK_MARGIN     = 0.05
LABEL_SMOOTHING = 0.05

# 样本权重
SAMPLE_WEIGHT_MODE  = "continuous"   # "continuous" | "threshold" | "none"
INTENSITY_THRESHOLD = 0.5
WEAK_SAMPLE_WEIGHT  = 0.1

# 被试筛选（留空表示全量）
TRAIN_SUBJECTS  = ""        # e.g. "1,2,3" 或 ""
VAL_SUBJECTS    = ""
TEST_SUBJECTS   = ""

# 训练容错
MAX_RUNTIME_HOURS = 10.0    # 超过此时长自动优雅退出
SAVE_INTERVAL     = 1      # 每 N epoch 保存一次

# 硬件
DEVICE = "auto"             # "auto" | "cuda" | "cpu"
USE_AMP = True

print(f"[CONFIG] model={MODEL_TYPE}, output={OUTPUT_DIR}, batch={BATCH_SIZE}")

---

## 🔐 Cell 3：ModelScope 登录（如需要）

In [ ]:
import os

token = MS_TOKEN or os.environ.get("MODELSCOPE_API_TOKEN", "")
if token:
    from modelscope import MsDataset
    print(f"[INFO] ModelScope token found (len={len(token)})")
else:
    print("[WARN] No ModelScope token — set MS_TOKEN in Cell 2 or export MODELSCOPE_API_TOKEN")
    print("[WARN] If data already exists locally at", LOCAL_DATA_PATH, "— skip Cell 4")

---

## 📥 Cell 4：下载预处理后数据

In [ ]:
from pathlib import Path
import os

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if Path(LOCAL_DATA_PATH).exists():
    print(f"[SKIP] 数据已存在: {LOCAL_DATA_PATH}")
else:
    print(f"[DOWNLOAD] {MS_DATASET_ID}/{MS_FILE_PATH} -> {LOCAL_DATA_PATH}")
    from modelscope.hub.api import HubApi
    api = HubApi(token=MS_TOKEN or os.environ.get("MODELSCOPE_API_TOKEN", ""))
    api.download_dataset_file(
        dataset_name=MS_DATASET_ID,
        file_path=MS_FILE_PATH,
        revision=MS_REVISION,
        target_path=LOCAL_DATA_PATH,
    )
    print(f"[OK] 下载完成: {LOCAL_DATA_PATH}")

import numpy as np
data = np.load(LOCAL_DATA_PATH, allow_pickle=True)
print(f"[DATA] X={data['X'].shape}, y={data['y'].shape}, s={data['s'].shape}")
print(f"[DATA] splits: {list(k for k in data.files if k.startswith('split_'))}")

---

## 🏋️ Cell 5：训练

In [ ]:
import sys, os
from pathlib import Path

# 确保 scripts 在 path 中
ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

# 构建训练命令
train_cmd = [
    "python", "scripts/train.py",
    "--data", LOCAL_DATA_PATH,
    "--output-dir", OUTPUT_DIR,
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--min-lr", str(MIN_LR),
    "--weight-decay", str(WEIGHT_DECAY),
    "--grad-clip", str(GRAD_CLIP),
    "--pretrain-epochs", str(PRETRAIN_EPOCHS),
    "--max-epochs", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--alpha-cls", str(ALPHA_CLS),
    "--beta-reg", str(BETA_REG),
    "--rank-margin", str(RANK_MARGIN),
    "--label-smoothing", str(LABEL_SMOOTHING),
    "--sample-weight-mode", SAMPLE_WEIGHT_MODE,
    "--intensity-threshold", str(INTENSITY_THRESHOLD),
    "--weak-sample-weight", str(WEAK_SAMPLE_WEIGHT),
    "--max-runtime-hours", str(MAX_RUNTIME_HOURS),
    "--save-interval", str(SAVE_INTERVAL),
    "--device", DEVICE,
    "--model-type", MODEL_TYPE,
]

if TRAIN_SUBJECTS: train_cmd += ["--train-subjects", TRAIN_SUBJECTS]
if VAL_SUBJECTS:   train_cmd += ["--val-subjects",   VAL_SUBJECTS]
if TEST_SUBJECTS:  train_cmd += ["--test-subjects",  TEST_SUBJECTS]
if not USE_AMP:    train_cmd.append("--no-amp")
else:              train_cmd.append("--amp")

print("[TRAIN CMD]", " ".join(train_cmd))
print("=" * 70)
import subprocess
result = subprocess.run(train_cmd, cwd=str(ROOT))
if result.returncode != 0:
    print(f"[ERROR] Training failed with exit code {result.returncode}")
else:
    print("[OK] Training completed")

---

## 🧠 Cell 6：编码特征

In [ ]:
from pathlib import Path

best_encoder = Path(OUTPUT_DIR) / "best_encoder.pt"
encode_output = Path(OUTPUT_DIR) / "encoded_features.npz"

if not best_encoder.exists():
    print(f"[WARN] Checkpoint not found: {best_encoder}")
    print("[WARN] Skipping encode step")
else:
    print(f"[ENCODE] checkpoint={best_encoder} -> {encode_output}")
    import subprocess
    encode_cmd = [
        "python", "scripts/encode.py",
        "--data",     LOCAL_DATA_PATH,
        "--checkpoint", str(best_encoder),
        "--output",   str(encode_output),
        "--model-type", MODEL_TYPE,
        "--batch-size", str(BATCH_SIZE),
        "--device", DEVICE,
    ]
    if USE_AMP: encode_cmd.append("--amp")
    result = subprocess.run(encode_cmd, cwd=str(ROOT))
    if result.returncode == 0:
        print(f"[OK] Encoded -> {encode_output}")
    else:
        print(f"[ERROR] Encode failed: {result.returncode}")

---

## 📊 Cell 7：结果汇总

In [ ]:
from pathlib import Path
import json, numpy as np

summary_path = Path(OUTPUT_DIR) / "summary.json"
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print("=" * 60)
    print(f"  模型类型   : {summary.get('model_type', MODEL_TYPE)}")
    print(f"  参数量     : {summary.get('n_params', 'N/A'):,} ({summary.get('n_params', 0)/1e6:.4f}M)")
    print(f"  训练 epoch : {summary.get('epochs_run', 'N/A')}")
    print(f"  最佳 val acc: {summary.get('best_val_acc', 'N/A'):.4f} @ ep {summary.get('best_epoch', 'N/A')}")
    print(f"  耗时       : {summary.get('elapsed_seconds', 0)/60:.1f} 分钟")
    test = summary.get("test", {})
    if test:
        print(f"  Test acc   : {test.get('acc', 'N/A'):.4f}")
        print(f"  Test MAE   : {test.get('intensity_mae', 'N/A'):.4f}")
    print("=" * 60)
else:
    print(f"[WARN] summary.json not found at {summary_path}")

# 编码结果（如果有）
encode_output = Path(OUTPUT_DIR) / "encoded_features.npz"
if encode_output.exists():
    feat = np.load(encode_output)
    print(f"\n[ENCODED] features={feat['features'].shape}, "
          f"acc={(feat['cls_pred'] == feat['labels']).mean():.4f}")
else:
    print("\n[INFO] No encoded features yet (run Cell 6 first)")

print("\n✅ Pipeline 完成！")
print(f"   输出目录: {OUTPUT_DIR}")
print(f"   最佳模型: {OUTPUT_DIR}/best_model.pt")
print(f"   编码器  : {OUTPUT_DIR}/best_encoder.pt")